# Modul 9: Big Data & Data Pipeline

**Project-Based Internship VINIX7**

Kelompok 3 (Universitas Sultan Ageng Tirtayasa)

Anggota Kelompok: Ahmad Jumhadi, Azhriler Lintang, Aura Salsa Azzahra

**Tahap 1:**

In [2]:
!pip install pyspark kaggle
!kaggle datasets download -d salmanabdu/tokopedia-product-reviews-2025
!unzip -o tokopedia-product-reviews-2025.zip

Dataset URL: https://www.kaggle.com/datasets/salmanabdu/tokopedia-product-reviews-2025
License(s): MIT
100% 3.54M/3.54M [00:00<00:00, 158MB/s]

Archive:  tokopedia-product-reviews-2025.zip
  inflating: tokopedia_product_reviews_2025.csv  


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Vinix7_Tokopedia_ETL") \
    .getOrCreate()

**Tahap 2:**

In [4]:
df_raw = spark.read.csv(
    "tokopedia_product_reviews_2025.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    quote='"',
    escape='"'
)

df_raw.printSchema()
df_raw.show(5, truncate=False)

print(f"Jumlah data sebelum cleaning: {df_raw.count()}")

root
 |-- review_text: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_variant: string (nullable = true)
 |-- product_price: integer (nullable = true)
 |-- product_url: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- rating: integer (nullable = true)
 |-- sold_count: integer (nullable = true)
 |-- shop_id: long (nullable = true)
 |-- sentiment_label: string (nullable = true)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+----------+--------------------------------------------------+-----------------+---------------+-------------+---------------------------------------------------------------------------------------------------------

In [5]:
from pyspark.sql.functions import col, sum

df_raw.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_raw.columns]).show()

print(f"Total Duplikat: {df_raw.count() - df_raw.dropDuplicates().count()}")

df_raw.describe(["rating", "product_price"]).show()

+-----------+-----------+---------+------------+----------------+---------------+-------------+-----------+----------+------+----------+-------+---------------+
|review_text|review_date|review_id|product_name|product_category|product_variant|product_price|product_url|product_id|rating|sold_count|shop_id|sentiment_label|
+-----------+-----------+---------+------------+----------------+---------------+-------------+-----------+----------+------+----------+-------+---------------+
|          0|          0|        0|           0|               0|          38794|            0|          0|         0|     0|         0|      0|              0|
+-----------+-----------+---------+------------+----------------+---------------+-------------+-----------+----------+------+----------+-------+---------------+

Total Duplikat: 0
+-------+-------------------+------------------+
|summary|             rating|     product_price|
+-------+-------------------+------------------+
|  count|              65543|

Berdasarkan hasil pengecekan, tidak ditemukan missing value maupun data duplikat pada dataset.
Hal ini menunjukkan bahwa kualitas data sudah cukup baik sehingga proses cleaning tidak
mengurangi jumlah data. Dengan kondisi ini, pipeline ETL dapat berjalan lebih efisien tanpa
kehilangan informasi penting.

**Interpretasi 1:**

Data ulasan mentah pelanggan ini lebih condong ke sistem pencatatan operasional (mendekati OLTP) karena data disimpan dalam bentuk transaksi detail dan terus bertambah secara real-time setiap kali pelanggan memberikan ulasan produk. Setiap baris data merepresentasikan aktivitas operasional individual, dan struktur seperti ini belum efisien untuk analisis skala besar.

Untuk keperluan data science, data perlu ditransformasikan terlebih dahulu menjadi bentuk OLAP melalui proses agregasi. Karena dengan agregasi, data yang awalnya sangat detail dapat diringkas menjadi informasi yang mudah dianalisis. Bentuk OLAP ini membantu proses analisis menjadi lebih cepat, efisien, dan mendukung pengambilan keputusan bisnis secara strategis.

**Tahap 3:**

In [6]:
from pyspark.sql.functions import to_date, count, avg, expr

df_clean = df_raw.dropna(subset=["review_text", "product_category", "sentiment_label"])
df_clean = df_clean.dropDuplicates()

print(f"Jumlah data sesudah cleaning: {df_clean.count()}")

df_formatted = df_clean.withColumn("review_date", to_date("review_date", "yyyy-MM-dd"))
df_formatted = df_formatted.withColumn("rating", expr("TRY_CAST(rating AS DOUBLE)"))

Jumlah data sesudah cleaning: 65543


In [7]:
df_mart = df_formatted.groupBy("product_category", "sentiment_label") \
    .agg(
        count("*").alias("total_reviews"),
        avg("rating").alias("average_rating")
    )

df_mart.show(5)
print(f"Jumlah baris Data Mart (Agregasi): {df_mart.count()}")

+------------------+---------------+-------------+------------------+
|  product_category|sentiment_label|total_reviews|    average_rating|
+------------------+---------------+-------------+------------------+
|Handphone & Tablet|       negative|           62|1.2096774193548387|
|       Pertukangan|        neutral|          147|               3.0|
|        Elektronik|       negative|           24|1.3333333333333333|
|          Olahraga|        neutral|          229|               3.0|
|       Pertukangan|       negative|          133|1.2932330827067668|
+------------------+---------------+-------------+------------------+
only showing top 5 rows
Jumlah baris Data Mart (Agregasi): 18


**Tahap 4:**

In [8]:
df_mart.write.mode("overwrite").parquet("tokopedia_sentiment_mart")

In [9]:
df_check = spark.read.parquet("tokopedia_sentiment_mart")
df_check.show()

+------------------+---------------+-------------+------------------+
|  product_category|sentiment_label|total_reviews|    average_rating|
+------------------+---------------+-------------+------------------+
|Handphone & Tablet|       negative|           62|1.2096774193548387|
|       Pertukangan|        neutral|          147|               3.0|
|        Elektronik|       negative|           24|1.3333333333333333|
|          Olahraga|        neutral|          229|               3.0|
|       Pertukangan|       negative|          133|1.2932330827067668|
| Makanan & Minuman|       negative|          263|1.3422053231939164|
|        Elektronik|        neutral|           21|               3.0|
|         Kesehatan|       negative|           39|1.2820512820512822|
|        Elektronik|       positive|         4157| 4.984123165744528|
|          Olahraga|       positive|        15094| 4.944083741884192|
|          Olahraga|       negative|          277|1.3249097472924187|
|         Kesehatan|

### **Interpretasi 2: Pseudocode**

**DAG:** `tokopedia_etl_pipeline`  
**Schedule:** `daily`

```text
[ START ]
    ↓
extract_data
  - Download dataset dari Kaggle
  - Unzip file CSV
    ↓
load_to_spark
  - Inisialisasi SparkSession
  - Load CSV ke Spark DataFrame (header=True, inferSchema=True)
    ↓
data_cleaning
  - Hapus data null (review_text, product_category, sentiment_label)
  - Hapus data duplikat
    ↓
data_transformation
  - Ubah review_date ke tipe Date
  - Cast rating ke Double
    ↓
data_aggregation
  - Group by product_category dan sentiment_label
  - Hitung total_reviews (count)
  - Hitung average_rating (avg)
    ↓
load_to_parquet
  - Simpan hasil agregasi ke format Parquet
  - Folder: tokopedia_sentiment_mart
    ↓
validation
  - Load ulang file Parquet
  - Validasi jumlah data dan struktur
    ↓
[ END ]

Jika pipeline ETL ini diotomatisasi menggunakan Apache Airflow dan dijalankan setiap malam, maka proses pertama yang dilakukan saat memulai pipeline adalah mengunduh dataset ulasan terbaru dari Kaggle. Setelah dataset berhasil diunduh, file ZIP akan diekstrak terlebih dahulu untuk mendapatkan file CSV. Selanjutnya, Airflow akan menginisialisasi SparkSession dan melakukan proses load file CSV ke dalam Spark DataFrame menggunakan opsi header=True dan inferSchema=True agar nama kolom serta tipe data dapat dikenali secara otomatis.

Tahap berikutnya adalah proses cleaning, yaitu menghapus data yang memiliki nilai null pada kolom penting (review_text, product_category, sentiment_label), serta menghapus data duplikat agar kualitas data tetap terjaga. Setelah data bersih, pipeline melanjutkan proses transformasi dengan mengubah format kolom tanggal menjadi tipe Date dan melakukan cast kolom rating menjadi tipe double.

Selanjutnya dilakukan proses agregasi data berdasarkan product_category dan sentiment_label untuk menghitung total_reviews menggunakan fungsi count() serta menghitung average_rating menggunakan fungsi avg() pada setiap kategori produk. Setelah proses transformasi selesai, hasil agregasi akan disimpan dalam format Parquet ke folder storage tokopedia_sentiment_mart.

Terakhir, Airflow dapat menjalankan task validasi dengan load ulang file Parquet untuk memastikan file output berhasil dibuat sebelum pipeline dinyatakan selesai.

**Interpretasi 3:**

Untuk kebutuhan Big data dan analisis data, format Parquet secara objektif lebih unggul dibandingkan dengan format CSV. Parquet menggunakan format penyimpanan berbasis kolom (columnar storage), sehingga proses pembacaan data menjadi lebih cepat karena sistem hanya membaca kolom yang diperlukan saat query dijalankan. Hal ini membuat performa analisis jauh lebih efisien dibanding CSV yang membaca seluruh data secara baris. Selain itu, ukuran file Parquet biasanya lebih kecil karena mendukung kompresi data yang lebih optimal. Dengan ukuran file yang lebih ringan, penggunaan storage menjadi lebih hemat dan proses transfer data juga lebih cepat. Parquet juga mampu menyimpan tipe data secara native, seperti integer, double, dan date, sehingga lebih aman dan konsisten untuk proses analitik. Hal ini berbeda dengan CSV yang sering membaca seluruh data sebagai string dan memerlukan proses parsing ulang.